# Part 3: Main 2SLS Analysis and Causal Inference

**Objective:**  
To estimate the causal effect of trade sanctions on the military expenditure of target states.

Naive OLS estimates are biased due to **endogeneity and reverse causality**: sanctions are often imposed on states already undergoing militarization. To overcome this, we implement a **Two-Stage Least Squares (2SLS)** identification strategy, utilizing the exogenous instruments derived in the previous steps (UNGA political distance and Gravity-based trade weights).

### Methodological Highlights:
1. **Zero-Stage Dyadic LPM:** We predict the probability of a sanction being imposed at the dyadic level (sender-target-year) using a Linear Probability Model (LPM) to avoid the "perfect separation" problem common in probit/logit models with rare events and high-dimensional fixed effects.
2. **Instrument Construction:** We aggregate these dyadic probabilities, weighting them by the exogenous structural trade shares (PPML gravity weights) to form the instrument for the First Stage.
3. **Custom TWFE Estimation:** We manually implement Two-Way Fixed Effects (Country and Year FEs) using the Frisch-Waugh-Lovell (FWL) theorem by demeaning the data. This allows for flexible and computationally efficient robust standard error estimation (HC1).

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
import time

warnings.filterwarnings('ignore')

# --- Configuration & File Paths ---
CLEANED_DIR = 'data/cleaned/'

GSDB_FILE = f'{CLEANED_DIR}cleaned_gsdb_dyadic.csv'
GEODIST_FILE = f'{CLEANED_DIR}cleaned_geodist_dyadic.csv'
UNGA_DIST_FILE = f'{CLEANED_DIR}cleaned_unga_distance_dyadic.csv'
GRAVITY_WEIGHTS_FILE = f'{CLEANED_DIR}gravity_trade_weights.csv'
OLS_TEMPLATE_FILE = f'{CLEANED_DIR}cleaned_sipri_country_year.csv' # Assuming base country-year df

USE_ENTITY_EFFECTS = True
USE_TIME_EFFECTS = True

## 1. Custom TWFE Demeaning Function (FWL Theorem)
To efficiently estimate the model with Country and Year Fixed Effects, we sweep out the fixed effects by subtracting the group means from all variables.

In [ ]:
def demean(df, columns, entity_col='iso3', time_col='year'):
    """
    Applies the Frisch-Waugh-Lovell (FWL) theorem to partial out
    Two-Way Fixed Effects (Entity and Time) by demeaning the variables.
    """
    df_demeaned = df.copy()

    if USE_ENTITY_EFFECTS:
        for col in columns:
            df_demeaned[col] = df_demeaned[col] - df_demeaned.groupby(entity_col)[col].transform('mean')

    if USE_TIME_EFFECTS:
        for col in columns:
            df_demeaned[col] = df_demeaned[col] - df_demeaned.groupby(time_col)[col].transform('mean')

    return df_demeaned

## 2. Zero-Stage Model (Dyadic Level)
We estimate the propensity of sender $i$ to sanction target $j$ in year $t$ using the sender's leave-one-out historical aggressiveness and the lagged UNGA voting distance.

In [ ]:
def prepare_dyadic_and_run_zero_stage(intensity='part', direction='exp'):
    """
    Constructs the dyadic dataset for a specific trade sanction type,
    estimates the Zero-Stage LPM, and generates gravity-weighted instruments.
    """
    print(f"\n--- Estimating Zero-Stage LPM for Trade {direction.upper()} ({intensity.upper()}) ---")

    # Load core dyadic data
    gsdb_df = pd.read_csv(GSDB_FILE)
    geodist_df = pd.read_csv(GEODIST_FILE)
    unga_dist_df = pd.read_csv(UNGA_DIST_FILE)
    trade_weights_df = pd.read_csv(GRAVITY_WEIGHTS_FILE)

    # Standardize years
    for df in [gsdb_df, unga_dist_df]:
        df['year'] = pd.to_datetime(df['year'], errors='coerce').dt.year.dropna().astype(int)

    col = f'trade_{direction}_{intensity}'

    # Create the full dyadic square panel
    unique_dyads = gsdb_df[['iso3_i', 'iso3_j']].drop_duplicates()
    all_years = pd.DataFrame({'year': sorted(gsdb_df['year'].unique())})
    dyadic_df = unique_dyads.merge(all_years, how='cross')
    dyadic_df = dyadic_df[dyadic_df['iso3_i'] != dyadic_df['iso3_j']]

    # Merge sanction indicators
    subset = gsdb_df[gsdb_df[col] == 1][['iso3_i', 'iso3_j', 'year', col]]
    dyadic_df = pd.merge(dyadic_df, subset, on=['iso3_i', 'iso3_j', 'year'], how='left')
    dyadic_df[col] = dyadic_df[col].fillna(0)

    # Calculate "Leave-one-out" Sender Aggressiveness
    aggr = gsdb_df[gsdb_df[col] == 1].groupby(['iso3_i', 'year'])[col].sum().reset_index()
    aggr = aggr.rename(columns={col: f'total_{col}'})
    dyadic_df = pd.merge(dyadic_df, aggr, on=['iso3_i', 'year'], how='left')
    dyadic_df[f'total_{col}'] = dyadic_df[f'total_{col}'].fillna(0)
    dyadic_df[f'log_aggr_{col}'] = np.log1p(dyadic_df[f'total_{col}'] - dyadic_df[col])

    # Merge Exogenous Predictors
    dyadic_df = pd.merge(dyadic_df, geodist_df, on=['iso3_i', 'iso3_j'], how='left')
    dyadic_df = pd.merge(dyadic_df, unga_dist_df, on=['iso3_i', 'iso3_j', 'year'], how='left')
    dyadic_df = pd.merge(dyadic_df, trade_weights_df, on=['iso3_i', 'iso3_j', 'year'], how='left')

    dyadic_df = dyadic_df.sort_values(by=['iso3_i', 'iso3_j', 'year'])

    # Create Lags
    dyadic_df[f'lag_aggr_{col}'] = dyadic_df.groupby(['iso3_i', 'iso3_j'])[f'log_aggr_{col}'].shift(1)
    dyadic_df['lag_vote_dist'] = dyadic_df.groupby(['iso3_i', 'iso3_j'])['vote_distance'].shift(1)
    dyadic_df[f'lag_inter_{col}'] = dyadic_df[f'lag_aggr_{col}'] * dyadic_df['lag_vote_dist']
    dyadic_df['ln_dist'] = np.log(dyadic_df['distance'].replace(0, 1))

    # Drop NAs for estimation
    cols_to_drop = ['lag_vote_dist', 'ln_dist', 'contiguity', 'common_language', 'colonial_link',
                    f'lag_aggr_{col}', f'lag_inter_{col}']
    dyadic_df.dropna(subset=cols_to_drop, inplace=True)

    # Estimate LPM (Zero-Stage)
    formula = f"{col} ~ lag_aggr_{col} + lag_vote_dist + lag_inter_{col} + ln_dist + contiguity + common_language + colonial_link"
    model_lpm = smf.ols(formula, data=dyadic_df).fit()
    print(f"LPM for {col} estimated successfully. R-squared: {model_lpm.rsquared:.3f}")

    # Generate gravity-weighted instrument
    weight_col = 'weight_export' if direction == 'exp' else 'weight_import'
    dyadic_df['weight_col'] = dyadic_df[weight_col].fillna(0)

    dyadic_df[f'prob_{col}'] = model_lpm.predict(dyadic_df)
    dyadic_df[f'weighted_prob_{col}'] = dyadic_df[f'prob_{col}'] * dyadic_df['weight_col']

    instr = dyadic_df.groupby(['iso3_j', 'year'])[f'weighted_prob_{col}'].sum().reset_index()
    instr = instr.rename(columns={'iso3_j': 'iso3', f'weighted_prob_{col}': f'pred_{col}'})

    return instr, col

## 3. First and Second Stage Estimation (2SLS)
We integrate the instrument into the country-year panel. The endogenous variable (real gravity-weighted sanction intensity) is instrumented by the predicted probabilities from the Zero-Stage. We evaluate both peacetime and wartime effects.

In [ ]:
def get_exog_matrix(df, cols):
    """Safely returns the exogenous matrix, adding a constant if FEs are disabled."""
    exog = df[cols]
    if not USE_ENTITY_EFFECTS and not USE_TIME_EFFECTS:
        exog = sm.add_constant(exog)
    return exog

def run_2sls_pipeline(instrument_df, sanction_col, direction):
    """
    Executes the First and Second stage of the 2SLS approach,
    accounting for heterogeneous effects during wartime.
    """
    print(f"\n{'='*70}\n=== 2SLS Estimation: {sanction_col.upper()} (Absolute Spending) ===\n{'='*70}")

    # Load Base Country-Year Panel
    # Note: Ensure ols_prepared_data includes basic controls (GDP, Polity, War)
    final_df = pd.read_csv(f'{CLEANED_DIR}ols_prepared_data.csv').sort_values(by=['iso3', 'year'])

    # Calculate Lagged Dependent Variable (Bureaucratic Inertia)
    final_df['lagged_ln_milex'] = final_df.groupby('iso3')['ln_milex_const_usd'].shift(1)

    # Process Endogenous Real Measure (Gravity-Weighted Sanctions)
    gsdb_df = pd.read_csv(GSDB_FILE)
    trade_weights_df = pd.read_csv(GRAVITY_WEIGHTS_FILE)

    weight_col = 'weight_export' if direction == 'exp' else 'weight_import'

    gsdb_subset = gsdb_df[gsdb_df[sanction_col] == 1][['iso3_i', 'iso3_j', 'year', sanction_col]]
    gsdb_subset = pd.merge(gsdb_subset, trade_weights_df, on=['iso3_i', 'iso3_j', 'year'], how='left')

    gsdb_subset[f'weighted_real'] = gsdb_subset[sanction_col] * gsdb_subset[weight_col].fillna(0)
    m_df = gsdb_subset.groupby(['iso3_j', 'year'])['weighted_real'].sum().reset_index()
    m_df = m_df.rename(columns={'iso3_j': 'iso3', 'weighted_real': f'real_{sanction_col}'})

    # Merge Data
    final_df = pd.merge(final_df, m_df, on=['iso3', 'year'], how='left')
    final_df[f'log_real_{sanction_col}'] = np.log1p(final_df[f'real_{sanction_col}'].fillna(0))

    final_df = pd.merge(final_df, instrument_df, on=['iso3', 'year'], how='left')
    final_df[f'log_pred_{sanction_col}'] = np.log1p(final_df[f'pred_{sanction_col}'].fillna(0))

    # Controls Setup
    dep_y = 'ln_milex_const_usd'
    exog_controls = ['ln_gdp_const_usd', 'war', 'polity', 'polity_sq', 'lagged_ln_milex']
    fs_exog_vars = [f'log_pred_{sanction_col}'] + exog_controls

    # Clean missing data before demeaning
    cols_to_clean = fs_exog_vars + [f'log_real_{sanction_col}', dep_y, 'iso3', 'year']
    work_df = final_df.dropna(subset=cols_to_clean).copy()

    # ---------------------------------------------------------
    # First Stage
    # ---------------------------------------------------------
    print("--- First Stage Diagnostics ---")
    fs_demeaned = demean(work_df, columns=fs_exog_vars + [f'log_real_{sanction_col}'])

    X_fs = get_exog_matrix(fs_demeaned, fs_exog_vars)
    fs_model = sm.OLS(fs_demeaned[f'log_real_{sanction_col}'], X_fs).fit(cov_type='HC1')

    f_stat = fs_model.f_test(f'log_pred_{sanction_col}=0').fvalue
    print(f"Weak Instrument F-statistic: {f_stat:.2f}")

    # Generate predictions (on demeaned data to preserve FWL logic)
    work_df[f'{sanction_col}_hat'] = fs_model.predict(X_fs)
    work_df[f'{sanction_col}_hat_peace'] = work_df[f'{sanction_col}_hat'] * (1 - work_df['war'])
    work_df[f'{sanction_col}_hat_war'] = work_df[f'{sanction_col}_hat'] * work_df['war']

    # ---------------------------------------------------------
    # Second Stage
    # ---------------------------------------------------------
    print("\n--- Second Stage Results ---")
    ss_vars = [dep_y, f'{sanction_col}_hat_peace', f'{sanction_col}_hat_war'] + exog_controls
    ss_demeaned = demean(work_df, columns=ss_vars)

    X_ss = get_exog_matrix(ss_demeaned, [f'{sanction_col}_hat_peace', f'{sanction_col}_hat_war'] + exog_controls)
    model_ss = sm.OLS(ss_demeaned[dep_y], X_ss).fit(cov_type='HC1')

    print(model_ss.summary().tables[1])

    try:
        p_val = model_ss.f_test(f'{sanction_col}_hat_peace = {sanction_col}_hat_war').pvalue
        print(f"\nHeterogeneity Test (Peace vs. War) p-value: {p_val:.4f}")
    except Exception:
        pass

## 4. Execution Pipeline
Now let's run the pipeline for Partial Export Sanctions (i.e. denying the target access to foreign goods).

In [ ]:
# Run the pipeline for Export Sanctions (Partial)
instr_exp_part, col_name = prepare_dyadic_and_run_zero_stage(intensity='part', direction='exp')
run_2sls_pipeline(instr_exp_part, col_name, direction='exp')